In [1]:
!pip install torch nltk

In [2]:
import json

intents = {
  "intents": [
    {"tag": "greeting",
     "patterns": ["Hi", "Hello", "Hey"],
     "responses": ["Hello!", "Hi there!", "Namaste!"]},

    {"tag": "goodbye",
     "patterns": ["Bye", "See you", "Goodbye"],
     "responses": ["Bye!", "See you later!", "Take care!"]},

    {"tag": "name",
     "patterns": ["What is your name", "Who are you"],
     "responses": ["I am a DL chatbot!", "You can call me AI Bot"]},
  ]
}

with open("intents.json", "w") as f:
    json.dump(intents, f)

In [3]:
import nltk
nltk.download('punkt')

from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenize(sentence):
    return nltk.word_tokenize(sentence)

def stem(word):
    return stemmer.stem(word.lower())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [5]:
import numpy as np
import nltk
nltk.download('punkt_tab')

all_words = []
tags = []
xy = []

with open('intents.json') as f:
    intents = json.load(f)

for intent in intents['intents']:
    tag = intent['tag']
    tags.append(tag)
    for pattern in intent['patterns']:
        w = tokenize(pattern)
        all_words.extend(w)
        xy.append((w, tag))

ignore_words = ['?', '!', '.', ',']
all_words = [stem(w) for w in all_words if w not in ignore_words]
all_words = sorted(set(all_words))
tags = sorted(set(tags))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
def bag_of_words(tokenized_sentence, words):
    sentence_words = [stem(w) for w in tokenized_sentence]
    bag = np.zeros(len(words), dtype=np.float32)

    for idx, w in enumerate(words):
        if w in sentence_words:
            bag[idx] = 1
    return bag

In [7]:
import torch
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

In [8]:
X_train = []
y_train = []

for (pattern_sentence, tag) in xy:
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    y_train.append(tags.index(tag))

X_train = np.array(X_train)
y_train = np.array(y_train)

X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train)

model = NeuralNet(len(all_words), 8, len(tags))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1000):
    outputs = model(X_train.float())
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training complete!")

Training complete!


In [9]:
def chat():
    print("Chatbot ready! (type 'quit' to exit)")
    while True:
        sentence = input("You: ")
        if sentence == "quit":
            break

        sentence = tokenize(sentence)
        X = bag_of_words(sentence, all_words)
        X = torch.from_numpy(X).float()

        output = model(X)
        _, predicted = torch.max(output, dim=0)

        tag = tags[predicted.item()]

        for intent in intents['intents']:
            if tag == intent["tag"]:
                print("Bot:", intent["responses"][0])

chat()

Chatbot ready! (type 'quit' to exit)
You: hello
Bot: Hello!
You: tell me about chatgpt
Bot: Hello!
You: how are you
Bot: I am a DL chatbot!
You: how are you
Bot: I am a DL chatbot!
You: i am abhishek
Bot: Hello!
You: quit
